# Final Validation Pass

**Source script:** `final_validation_pass.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import re
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

from run_eda_v3 import (
    BINARY_DISEASE_CLASSES,
    LGBMClassifier,
    SEED,
    add_moon_cyclic_features,
    build_feature_blocks,
    classify_columns,
    compute_balanced_sample_weight,
    derive_temporal_columns,
    infer_date_column,
    infer_outcome_column,
    load_dataset,
    make_hist_gb_binary_pipeline,
    make_lgbm_binary_pipeline,
)

ROOT = Path(__file__).resolve().parent
DEFAULT_INPUT = ROOT / "outputs" / "imputed_dataset_enriched.csv"
DEFAULT_OUTDIR = ROOT / "outputs" / "eda_v3" / "final_validation"
DEFAULT_CLINICAL_STATE_INPUT = ROOT / "outputs" / "eda_v3" / "clinical_threshold_extension" / "imputed_dataset_enriched_with_clinical_states.csv"
DEFAULT_SEVERITY_RULES = ROOT / "outputs" / "eda_v3" / "apriori_refined_final" / "severity_rules_clinical_thresholds.csv"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Final consolidation validation pass.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT, help="Base enriched dataset")
    parser.add_argument("--clinical-state-input", type=Path, default=DEFAULT_CLINICAL_STATE_INPUT, help="Dataset with clinical states/flags")
    parser.add_argument("--severity-rules", type=Path, default=DEFAULT_SEVERITY_RULES, help="Severity rules CSV")
    parser.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR, help="Output directory")
    parser.add_argument("--seed", type=int, default=SEED, help="Random seed")
    parser.add_argument("--corr-threshold", type=float, default=0.85, help="Correlation threshold")
    parser.add_argument("--perm-n", type=int, default=200, help="Permutation count for LGBM env-only")
    parser.add_argument(
        "--perm-jobs",
        type=int,
        default=1,
        help="Parallel jobs for permutation indices only (set >1 on HPC).",
    )
    return parser.parse_args()


def normalize_name(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def sanitize_item_name(text: str) -> str:
    out = re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")
    return out or "unknown"


def resolve_existing(path: Path, fallbacks: Sequence[Path], label: str) -> Path:
    if path.exists():
        return path
    for f in fallbacks:
        if f.exists():
            print(f"[info] {label} not found at {path}; using fallback: {f}")
            return f
    raise FileNotFoundError(f"{label} not found. Checked: {[str(path)] + [str(x) for x in fallbacks]}")


def correlation_filter_like_modeling(df: pd.DataFrame, weather_numeric_cols: Sequence[str], threshold: float) -> List[str]:
    if not weather_numeric_cols:
        return []
    X = df[list(weather_numeric_cols)].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    corr = X.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]
    return [c for c in weather_numeric_cols if c not in to_drop]


def build_model_aligned_feature_blocks(df: pd.DataFrame, corr_threshold: float) -> Tuple[pd.DataFrame, str, Dict[str, Dict[str, List[str]]]]:
    drop_weather_reason = [c for c in df.columns if "weather_missing_reason" in c.lower()]
    if drop_weather_reason:
        df = df.drop(columns=drop_weather_reason)

    outcome_col = infer_outcome_column(df)
    date_col = infer_date_column(df)
    df, temporal_cols = derive_temporal_columns(df, date_col)
    df, _ = add_moon_cyclic_features(df)
    col_groups = classify_columns(df, outcome_col, temporal_cols)
    kept_weather_numeric = correlation_filter_like_modeling(df, col_groups["weather_model_numeric"], corr_threshold)
    feature_blocks = build_feature_blocks(col_groups)
    env_numeric_original = set(col_groups["weather_model_numeric"])
    feature_blocks["environmental_only"]["numeric"] = sorted(kept_weather_numeric)
    feature_blocks["combined"]["numeric"] = sorted(
        [c for c in feature_blocks["combined"]["numeric"] if c not in env_numeric_original]
        + list(kept_weather_numeric)
    )
    return df, outcome_col, feature_blocks


def fit_predict_binary(
    X_train: pd.DataFrame,
    y_train_bin: pd.Series,
    X_eval: pd.DataFrame,
    model_family: str,
    numeric_cols: Sequence[str],
    categorical_cols: Sequence[str],
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    if model_family == "histgb":
        pipe = make_hist_gb_binary_pipeline(numeric_cols, categorical_cols, random_state=seed)
        sw = compute_balanced_sample_weight(y_train_bin)
        pipe.fit(X_train, y_train_bin, model__sample_weight=sw)
    elif model_family == "lgbm":
        pipe = make_lgbm_binary_pipeline(numeric_cols, categorical_cols, random_state=seed)
        pipe.fit(X_train, y_train_bin)
    else:
        raise ValueError(f"Unknown model_family: {model_family}")

    proba = pipe.predict_proba(X_eval)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return proba, pred


def run_holdout_histgb(
    df: pd.DataFrame,
    outcome_col: str,
    feature_blocks: Dict[str, Dict[str, List[str]]],
    seed: int,
) -> pd.DataFrame:
    y = df[outcome_col].astype(str)
    rows: List[Dict[str, object]] = []

    for block in ["environmental_only", "combined"]:
        num_cols = list(feature_blocks[block]["numeric"])
        cat_cols = list(feature_blocks[block]["categorical"])
        X = df[num_cols + cat_cols].copy()

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=seed,
            stratify=y,
        )

        for disease in BINARY_DISEASE_CLASSES:
            y_train_bin = (y_train == disease).astype(int)
            y_test_bin = (y_test == disease).astype(int)
            if y_train_bin.nunique() < 2 or y_test_bin.nunique() < 2:
                continue
            proba, pred = fit_predict_binary(
                X_train=X_train,
                y_train_bin=y_train_bin,
                X_eval=X_test,
                model_family="histgb",
                numeric_cols=num_cols,
                categorical_cols=cat_cols,
                seed=seed,
            )
            rows.append(
                {
                    "disease": disease,
                    "block": block,
                    "test_auc": float(roc_auc_score(y_test_bin, proba)),
                    "test_balanced_accuracy": float(balanced_accuracy_score(y_test_bin, pred)),
                }
            )
    return pd.DataFrame(rows).sort_values(["block", "disease"]).reset_index(drop=True)


def run_lgbm_env_cv_and_perm(
    df: pd.DataFrame,
    outcome_col: str,
    env_num: Sequence[str],
    env_cat: Sequence[str],
    seed: int,
    perm_n: int,
    perm_jobs: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    X = df[list(env_num) + list(env_cat)].copy()
    y = df[outcome_col].astype(str)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    splits = list(skf.split(X, y))

    cv_rows: List[Dict[str, object]] = []
    perm_rows: List[Dict[str, object]] = []

    for disease in BINARY_DISEASE_CLASSES:
        y_bin = (y == disease).astype(int).reset_index(drop=True)
        X_reset = X.reset_index(drop=True)

        fold_aucs: List[float] = []
        fold_baccs: List[float] = []
        for tr_idx, te_idx in splits:
            y_tr = y_bin.iloc[tr_idx]
            y_te = y_bin.iloc[te_idx]
            if y_tr.nunique() < 2 or y_te.nunique() < 2:
                continue
            proba, pred = fit_predict_binary(
                X_train=X_reset.iloc[tr_idx],
                y_train_bin=y_tr,
                X_eval=X_reset.iloc[te_idx],
                model_family="lgbm",
                numeric_cols=env_num,
                categorical_cols=env_cat,
                seed=seed,
            )
            fold_aucs.append(float(roc_auc_score(y_te, proba)))
            fold_baccs.append(float(balanced_accuracy_score(y_te, pred)))

        if not fold_aucs:
            continue
        obs_auc = float(np.mean(fold_aucs))
        obs_bacc = float(np.mean(fold_baccs))
        cv_rows.append(
            {
                "disease": disease,
                "block": "environmental_only",
                "cv_auc_mean": obs_auc,
                "cv_auc_std": float(np.std(fold_aucs, ddof=0)),
                "cv_balanced_accuracy_mean": obs_bacc,
                "cv_balanced_accuracy_std": float(np.std(fold_baccs, ddof=0)),
                "n_folds": len(fold_aucs),
            }
        )

        perm_aucs: List[float] = []
        y_arr = y_bin.to_numpy()
        def one_perm_mean_auc(i: int) -> float:
            rng = np.random.default_rng(seed + i)
            y_perm = pd.Series(rng.permutation(y_arr))
            p_fold_aucs: List[float] = []
            for tr_idx, te_idx in splits:
                y_tr = y_perm.iloc[tr_idx]
                y_te = y_perm.iloc[te_idx]
                if y_tr.nunique() < 2 or y_te.nunique() < 2:
                    continue
                proba, _ = fit_predict_binary(
                    X_train=X_reset.iloc[tr_idx],
                    y_train_bin=y_tr,
                    X_eval=X_reset.iloc[te_idx],
                    model_family="lgbm",
                    numeric_cols=env_num,
                    categorical_cols=env_cat,
                    seed=seed,
                )
                p_fold_aucs.append(float(roc_auc_score(y_te, proba)))
            if not p_fold_aucs:
                return np.nan
            return float(np.mean(p_fold_aucs))

        if perm_jobs > 1:
            raw_perm = Parallel(n_jobs=perm_jobs, backend="loky")(
                delayed(one_perm_mean_auc)(i) for i in range(perm_n)
            )
            perm_aucs = [float(v) for v in raw_perm if np.isfinite(v)]
        else:
            for i in range(perm_n):
                v = one_perm_mean_auc(i)
                if np.isfinite(v):
                    perm_aucs.append(float(v))

        perm_arr = np.asarray(perm_aucs, dtype=float)
        perm_mean = float(np.mean(perm_arr)) if perm_arr.size else np.nan
        perm_std = float(np.std(perm_arr, ddof=0)) if perm_arr.size else np.nan
        p_val = float((np.sum(perm_arr >= obs_auc) + 1.0) / (len(perm_arr) + 1.0)) if perm_arr.size else np.nan
        perm_rows.append(
            {
                "disease": disease,
                "block": "environmental_only",
                "n_perm": int(len(perm_arr)),
                "observed_auc": obs_auc,
                "perm_mean": perm_mean,
                "perm_std": perm_std,
                "p_value": p_val,
                "effect_size": float(obs_auc - perm_mean) if np.isfinite(perm_mean) else np.nan,
            }
        )

    cv_df = pd.DataFrame(cv_rows).sort_values("disease").reset_index(drop=True)
    perm_df = pd.DataFrame(perm_rows).sort_values("disease").reset_index(drop=True)
    return cv_df, perm_df


def parse_antecedents(text: str) -> List[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    return [x.strip() for x in text.split(",") if x.strip()]


def build_item_masker(
    df: pd.DataFrame,
    env_num: Sequence[str],
    env_cat: Sequence[str],
) -> Dict[str, object]:
    num_map = {sanitize_item_name(c): c for c in env_num if normalize_name(c) not in {"moonphasesin", "moonphasecos"}}
    cat_map = {sanitize_item_name(c): c for c in env_cat}
    return {"num_map": num_map, "cat_map": cat_map}


def categorical_series_like_mining(df: pd.DataFrame, col: str) -> pd.Series:
    s = df[col].astype(str).str.strip()
    s = s.replace({"": "missing", "nan": "missing", "None": "missing", "NaN": "missing"}).fillna("missing")
    return s


def item_mask(
    df: pd.DataFrame,
    item: str,
    masker: Dict[str, object],
    q_low: float,
    q_high: float,
) -> pd.Series:
    if item in {"moon_near_full", "moon_near_new"}:
        if item in df.columns:
            return df[item].fillna(False).astype(bool)
        return pd.Series(False, index=df.index)

    if item.endswith("_low") or item.endswith("_high"):
        mode = "low" if item.endswith("_low") else "high"
        base = item[: -len("_low")] if mode == "low" else item[: -len("_high")]
        col = masker["num_map"].get(base)
        if col is None:
            return pd.Series(False, index=df.index)
        s = pd.to_numeric(df[col], errors="coerce")
        valid = s.dropna()
        if valid.empty:
            return pd.Series(False, index=df.index)
        q = float(valid.quantile(q_low if mode == "low" else q_high))
        if mode == "low":
            return (s <= q).fillna(False)
        return (s >= q).fillna(False)

    if "__" in item:
        col_tok, val_tok = item.split("__", 1)
        col = masker["cat_map"].get(col_tok)
        if col is None:
            return pd.Series(False, index=df.index)
        s = categorical_series_like_mining(df, col)
        return s.map(lambda x: sanitize_item_name(x) == val_tok).fillna(False)

    return pd.Series(False, index=df.index)


def metrics_from_masks(ante: pd.Series, y: pd.Series) -> Tuple[float, float, float]:
    a = ante.fillna(False).astype(bool)
    yy = y.fillna(False).astype(bool)
    support = float((a & yy).mean())
    ante_prev = float(a.mean())
    base = float(yy.mean())
    conf = float(support / ante_prev) if ante_prev > 0 else np.nan
    lift = float(conf / base) if base > 0 and np.isfinite(conf) else np.nan
    return support, conf, lift


def classify_rule(full_lift: float, min_ablation_lift: float, sensitivity_lift: float) -> str:
    if full_lift > 1.5 and min_ablation_lift > 1.5 and sensitivity_lift > 1.5:
        return "Stable"
    if full_lift > 1.5 and (min_ablation_lift > 1.5 or sensitivity_lift > 1.5):
        return "Moderately sensitive"
    return "Unstable"


def run_apriori_negative_control(
    df_states: pd.DataFrame,
    env_num: Sequence[str],
    env_cat: Sequence[str],
    severity_rules_path: Path,
    outdir: Path,
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    rules_df = pd.read_csv(severity_rules_path)
    if "severity_marker" not in rules_df.columns:
        if "consequent_name" in rules_df.columns:
            rules_df = rules_df.rename(columns={"consequent_name": "severity_marker"})
        else:
            raise KeyError("Severity rules CSV must contain severity_marker or consequent_name.")


    bun_target = {
        "precipitation_hours_mean_7d_low",
        "precipitation_hours_peak_14d_low",
        "surface_pressure_max_day1_high",
    }
    rules_df["ante_list"] = rules_df["antecedents"].map(parse_antecedents)
    rules_df["ante_set"] = rules_df["ante_list"].map(set)
    rules_df = rules_df[
        ~(
            rules_df["severity_marker"].eq("BUN_increased")
            & rules_df["ante_set"].map(lambda s: s == bun_target)
        )
    ].copy()

    top10 = rules_df.sort_values("lift", ascending=False).head(10).copy().reset_index(drop=True)
    masker = build_item_masker(df_states, env_num, env_cat)

    out_rows: List[Dict[str, object]] = []
    for i, row in top10.iterrows():
        marker = str(row["severity_marker"])
        ants = list(row["ante_list"])
        if marker not in df_states.columns:
            continue
        y = df_states[marker].fillna(False).astype(bool)


        m_full = pd.Series(True, index=df_states.index)
        for it in ants:
            m_full = m_full & item_mask(df_states, it, masker, q_low=0.25, q_high=0.75)
        full_support, full_conf, full_lift = metrics_from_masks(m_full, y)


        m_sens = pd.Series(True, index=df_states.index)
        for it in ants:
            m_sens = m_sens & item_mask(df_states, it, masker, q_low=0.20, q_high=0.80)
        sens_support, sens_conf, sens_lift = metrics_from_masks(m_sens, y)


        abl_lifts: List[float] = []
        abl_supports: List[float] = []
        abl_confs: List[float] = []
        for drop_idx in range(len(ants)):
            m = pd.Series(True, index=df_states.index)
            for j, it in enumerate(ants):
                if j == drop_idx:
                    continue
                m = m & item_mask(df_states, it, masker, q_low=0.25, q_high=0.75)
            s, c, l = metrics_from_masks(m, y)
            abl_supports.append(s)
            abl_confs.append(c)
            abl_lifts.append(l)

        min_ablation_lift = float(np.nanmin(abl_lifts)) if abl_lifts else np.nan
        max_ablation_lift = float(np.nanmax(abl_lifts)) if abl_lifts else np.nan
        classification = classify_rule(full_lift, min_ablation_lift, sens_lift)

        out_rows.append(
            {
                "rank_by_original_lift": int(i + 1),
                "severity_marker": marker,
                "antecedents": ", ".join(ants),
                "n_antecedents": len(ants),
                "original_lift": float(row["lift"]),
                "manual_lift_25_75": full_lift,
                "manual_confidence_25_75": full_conf,
                "manual_support_25_75": full_support,
                "ablation_best_lift": max_ablation_lift,
                "ablation_min_lift": min_ablation_lift,
                "sensitivity_lift_20_80": sens_lift,
                "sensitivity_confidence_20_80": sens_conf,
                "sensitivity_support_20_80": sens_support,
                "classification": classification,
            }
        )

    robust_df = pd.DataFrame(out_rows).sort_values("rank_by_original_lift").reset_index(drop=True)
    robust_df.to_csv(outdir / "apriori_top10_robustness.csv", index=False)

    summary_counts = {
        "n_stable": int((robust_df["classification"] == "Stable").sum()) if not robust_df.empty else 0,
        "n_moderately_sensitive": int((robust_df["classification"] == "Moderately sensitive").sum()) if not robust_df.empty else 0,
        "n_unstable": int((robust_df["classification"] == "Unstable").sum()) if not robust_df.empty else 0,
    }
    with (outdir / "apriori_robustness_summary.txt").open("w", encoding="utf-8") as f:
        f.write(f"n_stable={summary_counts['n_stable']}\n")
        f.write(f"n_moderately_sensitive={summary_counts['n_moderately_sensitive']}\n")
        f.write(f"n_unstable={summary_counts['n_unstable']}\n")

    return robust_df, summary_counts


def classify_signal_strength(mean_auc: float, n_sig: int) -> str:
    if mean_auc >= 0.65 and n_sig >= 2:
        return "Strong"
    if mean_auc >= 0.55 and n_sig >= 1:
        return "Moderate"
    return "Weak"


def main() -> None:
    args = parse_args()
    np.random.seed(args.seed)
    args.outdir.mkdir(parents=True, exist_ok=True)

    if LGBMClassifier is None:
        raise ImportError(
            "Missing dependency 'lightgbm'. Install in this environment before running final_validation_pass.py."
        )
    print(f"Permutation jobs: {max(1, int(args.perm_jobs))}")
    if args.perm_jobs > 1:
        print("Reminder: set OMP_NUM_THREADS=1 to avoid oversubscription.")


    base_df_raw = load_dataset(args.input)
    base_df, outcome_col, feature_blocks = build_model_aligned_feature_blocks(base_df_raw, args.corr_threshold)
    env_num = list(feature_blocks["environmental_only"]["numeric"])
    env_cat = list(feature_blocks["environmental_only"]["categorical"])


    holdout_df = run_holdout_histgb(base_df, outcome_col, feature_blocks, seed=args.seed)
    holdout_df.to_csv(args.outdir / "holdout_results_binary.csv", index=False)


    lgbm_cv_df, lgbm_perm_df = run_lgbm_env_cv_and_perm(
        df=base_df,
        outcome_col=outcome_col,
        env_num=env_num,
        env_cat=env_cat,
        seed=args.seed,
        perm_n=args.perm_n,
        perm_jobs=max(1, int(args.perm_jobs)),
    )
    lgbm_cv_df.to_csv(args.outdir / "lgbm_cv_binary_env_only.csv", index=False)
    lgbm_perm_df.to_csv(args.outdir / "lgbm_permtest_env_only.csv", index=False)


    hist_cv_path = resolve_existing(
        ROOT / "outputs" / "eda_v3" / "tables" / "cv_binary_metric_summary.csv",
        fallbacks=[],
        label="HistGB CV summary",
    )
    hist_perm_path = resolve_existing(
        ROOT / "outputs" / "eda_v3" / "tables" / "permtest_envonly_auc_summary.csv",
        fallbacks=[],
        label="HistGB permutation summary",
    )
    hist_cv = pd.read_csv(hist_cv_path)
    hist_perm = pd.read_csv(hist_perm_path)
    hist_cv_env = hist_cv[hist_cv["feature_block"] == "environmental_only"][["disease", "roc_auc_mean"]].rename(columns={"roc_auc_mean": "histgb_auc"})
    hist_perm_sub = hist_perm[["disease", "p_value"]].rename(columns={"p_value": "histgb_perm_p"})
    lgbm_cv_sub = lgbm_cv_df[["disease", "cv_auc_mean"]].rename(columns={"cv_auc_mean": "lgbm_auc"})
    lgbm_perm_sub = lgbm_perm_df[["disease", "p_value"]].rename(columns={"p_value": "lgbm_perm_p"})
    comp = hist_cv_env.merge(lgbm_cv_sub, on="disease", how="outer").merge(hist_perm_sub, on="disease", how="left").merge(lgbm_perm_sub, on="disease", how="left")
    comp["direction_consistent"] = np.sign(comp["histgb_auc"] - 0.5) == np.sign(comp["lgbm_auc"] - 0.5)
    comp.to_csv(args.outdir / "model_family_comparison.csv", index=False)


    clinical_state_input = resolve_existing(
        args.clinical_state_input,
        fallbacks=[ROOT / "outputs" / "eda_v3" / "clinical_threshold_extension" / "imputed_dataset_enriched_with_clinical_states.csv"],
        label="clinical state dataset",
    )
    severity_rules_path = resolve_existing(
        args.severity_rules,
        fallbacks=[
            ROOT / "outputs" / "eda_v3" / "clinical_threshold_extension" / "severity_rules_clinical_thresholds.csv",
            ROOT / "outputs" / "eda_v3" / "apriori_refined_final" / "severity_rules.csv",
        ],
        label="severity rules",
    )
    df_states = pd.read_csv(clinical_state_input)
    robust_df, robust_counts = run_apriori_negative_control(
        df_states=df_states,
        env_num=env_num,
        env_cat=env_cat,
        severity_rules_path=severity_rules_path,
        outdir=args.outdir,
    )


    holdout_env_mean_auc = float(holdout_df[holdout_df["block"] == "environmental_only"]["test_auc"].mean()) if not holdout_df.empty else np.nan
    lgbm_env_mean_auc = float(lgbm_cv_df["cv_auc_mean"].mean()) if not lgbm_cv_df.empty else np.nan
    hist_sig = int((comp["histgb_perm_p"] < 0.05).sum()) if not comp.empty else 0
    lgbm_sig = int((comp["lgbm_perm_p"] < 0.05).sum()) if not comp.empty else 0
    env_signal = classify_signal_strength(float(np.nanmean([holdout_env_mean_auc, lgbm_env_mean_auc])), min(hist_sig, lgbm_sig))

    total_rules = max(1, robust_counts["n_stable"] + robust_counts["n_moderately_sensitive"] + robust_counts["n_unstable"])
    stable_frac = robust_counts["n_stable"] / total_rules
    if stable_frac >= 0.5:
        severity_signal = "Strong"
    elif (robust_counts["n_stable"] + robust_counts["n_moderately_sensitive"]) / total_rules >= 0.5:
        severity_signal = "Moderate"
    else:
        severity_signal = "Weak"

    if env_signal == "Strong" and severity_signal in {"Strong", "Moderate"}:
        overall_conf = "High"
    elif env_signal in {"Moderate", "Strong"} and severity_signal in {"Moderate", "Weak"}:
        overall_conf = "Moderate"
    else:
        overall_conf = "Low"

    md: List[str] = []
    md.append("# Final Validation Summary")
    md.append("")
    md.append("## 1) Holdout vs CV Direction")
    md.append(f"- Holdout env-only mean AUC: {holdout_env_mean_auc:.4f}")
    md.append(f"- LightGBM env-only CV mean AUC: {lgbm_env_mean_auc:.4f}")
    if not comp.empty:
        agree = int(comp["direction_consistent"].fillna(False).sum())
        md.append(f"- HistGB vs LightGBM directional consistency: {agree}/{len(comp)} diseases.")
    md.append("")
    md.append("## 2) Model-family replication")
    if comp.empty:
        md.append("- Comparison table unavailable.")
    else:
        for _, r in comp.iterrows():
            md.append(
                f"- {r['disease']}: HistGB AUC={r['histgb_auc']:.4f}, LGBM AUC={r['lgbm_auc']:.4f}, "
                f"HistGB p={r['histgb_perm_p']:.4f}, LGBM p={r['lgbm_perm_p']:.4f}, "
                f"direction_consistent={bool(r['direction_consistent'])}"
            )
    md.append("")
    md.append("## 3) Apriori robustness (top10 negative control)")
    md.append(f"- Stable: {robust_counts['n_stable']}")
    md.append(f"- Moderately sensitive: {robust_counts['n_moderately_sensitive']}")
    md.append(f"- Unstable: {robust_counts['n_unstable']}")
    md.append("")
    md.append("## 4) Final statement")
    md.append(f"- Environmental disease classification signal: **{env_signal}**")
    md.append(f"- Environmental severity modulation signal: **{severity_signal}**")
    md.append(f"- Overall robustness confidence: **{overall_conf}**")
    (args.outdir / "final_validation_summary.md").write_text("\n".join(md), encoding="utf-8")

    print(f"Holdout mean AUC (env-only): {holdout_env_mean_auc:.4f}")
    print(f"LightGBM mean AUC (env-only): {lgbm_env_mean_auc:.4f}")
    print(f"# Stable Apriori rules: {robust_counts['n_stable']}")
    print(f"Final robustness confidence: {overall_conf}")


if __name__ == "__main__":
    main()
